# ExploreKit — end-to-end demo

Notebook только демонстрирует готовую библиотеку. Основная логика остаётся в `.py` модулях.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "configs").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
ROOT


WindowsPath('c:/Users/user/Downloads/Telegram Desktop/ek-3-3/ek-3')

## 1. Smoke: policy → DecisionLog → OPE

Используется реальная epsilon-greedy политика участника 1, логирование участника 4 и IPS участника 2.

In [2]:
from explorekit.integration import run_smoke

smoke = run_smoke(seed=42, n_rounds=1000)
{
    "n_logs": smoke.n_logs,
    "mean_reward": smoke.mean_reward,
    "ips": smoke.ips_estimate,
    "ess": smoke.ess,
    "reliability": smoke.reliability,
}


{'n_logs': 1000,
 'mean_reward': 0.055,
 'ips': 0.055,
 'ess': 1000.0,
 'reliability': 'Reliable'}

## 2. End-to-end experiment

Для notebook уменьшаем размер только ради быстрого демонстрационного запуска; параметры финального CLI остаются в YAML.

In [3]:
from experiments.run_experiment import run_experiment
from explorekit.integration import load_config

config = load_config(ROOT / "configs" / "moderate.yaml")
config["simulation"]["n_rounds"] = 1000
config["bootstrap"]["n_bootstrap"] = 100
result = run_experiment(config, save=False)
result


ExperimentResult(experiment_name='ExploreKit moderate exploration', goal='Compare true CTR, OPE estimates and exploration/cold-item metrics.', preset_name='moderate', logging_policy='epsilon_greedy_logging', evaluation_policy='epsilon_greedy(eps=0.1)', n_rounds=1000, seed=42, parameters={'policy': {'n_actions': 80, 'min_epsilon': 0.0, 'max_epsilon': 1.0, 'epsilon': 0.1, 'whitelist': None, 'blacklist': None, 'exploration_subset': None}, 'simulation': {'n_rounds': 1000, 'base_score_noise': 0.3, 'reward_model_noise': 0.25, 'logging_epsilon': 0.5, 'cold_penalty': 1.2, 'target_ctr': 0.05}, 'cold_start': {'max_age_days': 7, 'min_impressions': 20, 'boost_multiplier': 2.0, 'time_to_n': 20}, 'ope': {'max_weight': 15.0}, 'bootstrap': {'n_bootstrap': 100, 'confidence_level': 0.95}}, true_ctr=0.044776503359157005, expected_ctr=0.05281124006373762, expected_ctr_estimator='DR', expected_ci_low=0.0384926611406252, expected_ci_high=0.06849663880631475, ips_estimate=0.053532426862327585, ips_ci_low=0.0

In [4]:
import pandas as pd

pd.DataFrame([{
    "preset": result.preset_name,
    "policy": result.evaluation_policy,
    "true_ctr": result.true_ctr,
    "ips": result.ips_estimate,
    "snips": result.snips_estimate,
    "dr": result.dr_estimate,
    "ess": result.ess,
    "reliability": result.reliability,
    "exploration_share": result.exploration_share,
    "cold_item_speedup": result.cold_item_speedup,
    "ctr_cost": result.ctr_cost,
}])


,preset,policy,true_ctr,ips,snips,dr,ess,reliability,exploration_share,cold_item_speedup,ctr_cost
0,moderate,epsilon_greedy(eps=0.1),0.044777,0.053532,0.053029,0.052811,620.952405,Reliable,0.097,1.052967,0.00525


## 3. Симулятор и Cold-Item Booster

Каталог, скрытая вероятность клика и сравнение пресетов живут в `explorekit.simulator`. Notebook только вызывает готовые функции.

In [8]:
import numpy as np
from explorekit.simulator import ItemCatalog, make_replication
from explorekit.policies import EpsilonGreedyPolicy
from experiments.cold_item_experiment import run_cold_item_experiment

rng = np.random.default_rng(42)
catalog = ItemCatalog(n_actions=12, feature_dim=4, rng=rng)
print("warm items:", catalog.n_warm)
print("available day 0:", catalog.available_at(0).tolist())
print("available day 10:", catalog.available_at(10).tolist())

policy = EpsilonGreedyPolicy(
    n_actions=12, epsilon=0.1, min_epsilon=0.0, max_epsilon=1.0, seed=42
)
rep = make_replication(seed=42, n_rounds=200, policy=policy, time_to_n_threshold=8)
print({
    "true_ctr": round(rep.true_value, 4),
    "speedup": None if rep.cold_item_speedup is None else round(rep.cold_item_speedup, 3),
    "ctr_cost": None if rep.ctr_cost is None else round(rep.ctr_cost, 4),
    "n_logs": len(rep.logs),
    "chosen_item_type": type(rep.logs[0].chosen_item).__name__,
})

run_cold_item_experiment(n_rounds=160)

warm items: 9
available day 0: [0, 1, 2, 3, 4, 5, 6, 7, 8]
available day 10: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
{'true_ctr': 0.0552, 'speedup': 1.23, 'ctr_cost': 0.0014, 'n_logs': 200, 'chosen_item_type': 'int'}


,preset,epsilon,seed,n_rounds,true_ctr,served_ctr,ctr_cost,cold_item_impressions,time_to_n,reached_n_eval,speedup,exploration_share
0,baseline,0.00,42,160,0.055400,0.055400,0.000000,0,160.0,0.0,1.0,0.00000
1,conservative,0.05,42,160,0.052879,0.052638,0.002762,34,160.0,0.0,1.0,0.02500
2,moderate,0.10,42,160,0.050357,0.051422,0.003978,36,160.0,0.0,1.0,0.04375
3,aggressive,0.25,42,160,0.042794,0.044489,0.010911,46,160.0,0.0,1.0,0.17500
